# Correlation & auth checks — M2-19/M3-17, M3-10, M3-2, M3-27

Four cases, all in the same family: reject a callback that doesn't genuinely correlate to something this
server actually did, or that isn't genuinely from ABDM — while still degrading gracefully (no crash, still
acks) when a callback is legitimate but references something we have no record of.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M2-19 / M3-17 — every M2/M3 callback route requires a real ABDM-signed bearer JWT

**Real-world scenario:** before this fix, every callback route in `server/callbacks/router.py` accepted an
unauthenticated POST. Anyone who discovered or guessed a callback URL (e.g. `/api/v3/consent/request/hip/notify`)
could send a fabricated payload and have it processed exactly as if ABDM had sent it — no signature, no
proof it came from ABDM at all.

**Test approach:** `verify_abdm_callback()` is the one shared FastAPI dependency wired into every M2/M3
route, so testing it once covers both tracker cases. No real network call to ABDM's JWKS endpoint is made —
the JWKS client is stubbed so this stays fast, offline, and repeatable; the *positive* case still exercises
the real `jwt.decode()` signature-verification path, just against a locally generated test keypair standing
in for ABDM's real signing key (this proves the verification logic itself works, not that we can forge
ABDM's actual private key — we can't, and that's the point).

**Pass criteria:** a request with no `Authorization` header is rejected (401); a request with a garbage/
unverifiable token is rejected (401); a request carrying a validly-signed token with the right claims
(`iss`, `azp`, `aud`, unexpired) is accepted.

In [2]:
import time
from unittest.mock import patch

import server.callbacks.utils.jwt_auth as jwt_auth
from fastapi import HTTPException
import jwt as pyjwt
from jwt import PyJWKClientError
from cryptography.hazmat.primitives.asymmetric import rsa


class FakeURL:
    path = "/api/v3/consent/request/hip/notify"

class FakeRequest:
    def __init__(self, headers):
        self.headers = headers
        self.url = FakeURL()


# Test 1: no Authorization header at all.
req_no_auth = FakeRequest({})
try:
    await jwt_auth.verify_abdm_callback(req_no_auth)
    harness.check("no-auth-header request rejected with 401", False)
except HTTPException as exc:
    harness.check("no-auth-header request rejected with 401", exc.status_code == 401)

# Test 2: garbage/unresolvable token -- stub the JWKS client so this needs no real network call.
with patch.object(jwt_auth._jwks_client, "get_signing_key_from_jwt", side_effect=PyJWKClientError("no matching key")):
    req_garbage = FakeRequest({"authorization": "Bearer not-a-real-token"})
    try:
        await jwt_auth.verify_abdm_callback(req_garbage)
        harness.check("garbage-token request rejected with 401", False)
    except HTTPException as exc:
        harness.check("garbage-token request rejected with 401", exc.status_code == 401)

# Test 3: positive control -- a validly-signed token, self-signed with a locally generated RSA
# keypair standing in for ABDM's real signing key. Exercises the REAL jwt.decode() verification path.
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
claims = {"iss": jwt_auth.ABDM_ISSUER, "azp": "gateway", "aud": "account", "exp": int(time.time()) + 300}
token = pyjwt.encode(claims, private_key, algorithm="RS256")

class FakeSigningKey:
    def __init__(self, key):
        self.key = key

with patch.object(jwt_auth._jwks_client, "get_signing_key_from_jwt", return_value=FakeSigningKey(private_key.public_key())):
    req_valid = FakeRequest({"authorization": f"Bearer {token}"})
    try:
        await jwt_auth.verify_abdm_callback(req_valid)
        harness.check("validly-signed token with correct claims is accepted", True)
    except HTTPException:
        harness.check("validly-signed token with correct claims is accepted", False)


2026-08-14 18:41:50     [ERROR] Rejected callback to /api/v3/consent/request/hip/notify: missing or malformed Authorization header
2026-08-14 18:41:50     [ERROR] Rejected callback to /api/v3/consent/request/hip/notify: Could not resolve a signing key for this token: no matching key


PASS -- no-auth-header request rejected with 401
PASS -- garbage-token request rejected with 401
PASS -- validly-signed token with correct claims is accepted


---
## M3-10 — an on-fetch callback must correlate back to a fetch we actually made

**Real-world scenario:** the on-fetch route now requires a valid ABDM JWT (above), which blocks any
outside stranger. But that alone doesn't stop ABDM (or a replay/mixup) from delivering an on-fetch callback
for a `consentId` we never actually initiated a fetch for, or for the wrong `consentId` under a `requestId`
we do recognize. Before this fix, any authenticated callback was stored unconditionally.

**Pass criteria:** a fabricated `requestId` (no matching pending fetch) is rejected; a real `requestId`
but mismatched `consentId` is rejected; a genuine, fully-correlated callback is stored.

In [3]:
import server.callbacks.services.consent_hiu_on_fetch_service as on_fetch_service
from server.callbacks.repository.pending_consent_request_repository import save_pending_consent_request
from server.callbacks.repository.hiu_consent_repository import get_hiu_consent

harness.activate_scratch_storage("m3_10")

save_pending_consent_request("req-fetch-real", {"consent_id": "consent-real", "hiu_id": "HIU-1"})

body_fabricated = {"consent": {"status": "GRANTED", "consentDetail": {"consentId": "consent-real"}}, "response": {"requestId": "req-NEVER-MADE"}}
await on_fetch_service.process_consent_hiu_on_fetch({"body": body_fabricated})
harness.check("fabricated/unknown requestId is rejected, not stored", get_hiu_consent("consent-real") is None)

body_mismatched = {"consent": {"status": "GRANTED", "consentDetail": {"consentId": "consent-DIFFERENT"}}, "response": {"requestId": "req-fetch-real"}}
await on_fetch_service.process_consent_hiu_on_fetch({"body": body_mismatched})
harness.check("real requestId but mismatched consentId is rejected, not stored", get_hiu_consent("consent-DIFFERENT") is None)

body_genuine = {"consent": {"status": "GRANTED", "consentDetail": {"consentId": "consent-real"}, "signature": "sig"}, "response": {"requestId": "req-fetch-real"}}
with patch.object(on_fetch_service, "maybe_trigger_health_information_request", lambda *a, **kw: None):
    await on_fetch_service.process_consent_hiu_on_fetch({"body": body_genuine})
harness.check("genuine, fully-correlated callback IS stored", get_hiu_consent("consent-real") is not None)


2026-08-14 18:42:43  -> Full consent artefact received from ABDM (POST /api/v3/hiu/consent/on-fetch)
2026-08-14 18:42:43     [ERROR] on-fetch callback for consentId=consent-real has no matching pending fetch (requestId='req-NEVER-MADE') -- rejecting, not storing.
2026-08-14 18:42:43  -> Full consent artefact received from ABDM (POST /api/v3/hiu/consent/on-fetch)
2026-08-14 18:42:43     [ERROR] on-fetch callback consentId=consent-DIFFERENT does not match the consentId='consent-real' we actually requested for requestId=req-fetch-real -- rejecting, not storing.
2026-08-14 18:42:43  -> Full consent artefact received from ABDM (POST /api/v3/hiu/consent/on-fetch)
2026-08-14 18:42:43  -> Consent artefact consent-real stored -- Block 1 complete for this consent.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_10_26jt2h8g
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- fabricated/unknown requestId is rejected, not stored
PASS -- real requestId but mismatched consentId is rejected, not stored
PASS -- genuine, fully-correlated callback IS stored


True

---
## M3-2 — a transactionId already linked to a different pending request refuses to relink

**Real-world scenario:** two Health Information Requests are in flight at once, each its own `requestId`
and its own freshly generated ECDH key material. If a second on-request callback ever claimed the SAME
`transactionId` as an already-linked, different `requestId`'s session, the old unconditional `link_transaction_id()`
call would silently repoint that transactionId at the new session — so the later data push (which arrives
keyed only by `transactionId`) would resolve to the WRONG session's key material, and a genuinely correct
push would fail to decrypt.

**Pass criteria:** the first, genuine link succeeds; a later callback claiming the same transactionId for a
different requestId is rejected and logged; the original linkage is untouched.

In [4]:
import server.callbacks.services.health_information_hiu_on_request_service as on_request_service
from server.callbacks.repository.pending_health_information_request_repository import (
    save_pending_health_information_request as save_pending_hir,
    get_request_id_for_transaction_id,
)

harness.activate_scratch_storage("m3_2")

save_pending_hir("request-A", {"key_material": {"nonce": "A"}})
save_pending_hir("request-B", {"key_material": {"nonce": "B"}})

await on_request_service.process_health_information_hiu_on_request({
    "body": {"response": {"requestId": "request-A"}, "hiRequest": {"transactionId": "txn-shared", "sessionStatus": "REQUESTED"}}
})
harness.check("txn-shared correctly linked to request-A", get_request_id_for_transaction_id("txn-shared") == "request-A")

await on_request_service.process_health_information_hiu_on_request({
    "body": {"response": {"requestId": "request-B"}, "hiRequest": {"transactionId": "txn-shared", "sessionStatus": "REQUESTED"}}
})
harness.check("collision attempt rejected -- still resolves to request-A, not overwritten by request-B", get_request_id_for_transaction_id("txn-shared") == "request-A")


2026-08-14 18:44:20  -> Data request acknowledged by ABDM (POST /api/v3/hiu/health-information/on-request)
2026-08-14 18:44:20  -> transactionId txn-shared recorded for requestId request-A (sessionStatus=REQUESTED)
2026-08-14 18:44:20     [WAITING] Waiting for the HIP to push encrypted records directly to our dataPushUrl
2026-08-14 18:44:20  -> Data request acknowledged by ABDM (POST /api/v3/hiu/health-information/on-request)
2026-08-14 18:44:20     [ERROR] transactionId txn-shared is already linked to a different pending requestId ('request-A') -- refusing to relink it to requestId 'request-B'. Ignoring this on-request callback as a probable cross-request mixup/replay rather than crossing two sessions' encryption keys.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_2_00lauovv
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- txn-shared correctly linked to request-A
PASS -- collision attempt rejected -- still resolves to request-A, not overwritten by request-B


True

---
## M3-27 — a notify callback for an unknown consentRequestId degrades gracefully (CONFIRM ONLY)

**Real-world scenario:** ABDM sends a `notify` (GRANTED) callback whose `consentRequestId` doesn't match
any pending record here — e.g. state was lost, or this is a genuinely stray/misdirected message.
`consent_hiu_notify_service.py` already handles this: it logs the gap, skips `fetch_consent()` (there's no
`hiu_id` to safely use), and still acks ABDM with whatever it has, rather than crashing or guessing at a
`hiu_id`. No fix needed — this cell proves that's real.

**Pass criteria:** no crash; `fetch_consent()` is never called; ABDM still gets an ack.

In [5]:
import server.callbacks.services.consent_hiu_notify_service as notify_service

harness.activate_scratch_storage("m3_27")

ack_recorder = harness.CallRecorder(harness.FakeResponse(202))
body_unknown = {"notification": {"consentRequestId": "cr-NEVER-INITIATED", "status": "GRANTED", "consentArtefacts": [{"id": "artefact-1"}]}}

with patch.object(notify_service, "send_consent_hiu_on_notify", ack_recorder), \
     patch.object(notify_service, "fetch_consent") as fetch_mock:
    try:
        await notify_service.process_consent_hiu_notify({"headers": {"request-id": "req-notify-1"}, "body": body_unknown})
        no_crash = True
    except Exception:
        no_crash = False

harness.check("unknown consentRequestId does not crash the handler", no_crash)
harness.check("fetch_consent() is never called (no hiu_id resolvable)", fetch_mock.call_count == 0)
harness.check("ABDM still gets acked despite the missing pending record", ack_recorder.call_count == 1)


2026-08-14 18:45:51  -> Consent status notification received from ABDM (POST /api/v3/hiu/consent/request/notify)
2026-08-14 18:45:51  -> Extracted consentRequestId and status (GRANTED) -- 1 artefact(s)
2026-08-14 18:45:51     [ERROR] No pending consent request found for consentRequestId cr-NEVER-INITIATED -- cannot resolve our hiu_id.
2026-08-14 18:45:51     [ERROR] Skipping fetch_consent() for 1 artefact(s) -- no hiu_id resolved for consentRequestId cr-NEVER-INITIATED.
2026-08-14 18:45:51     [API] Acknowledging Consent Notification to ABDM -- POST .../hiu/on-notify -> 202
2026-08-14 18:45:51     [WAITING] Waiting for ABDM's on-fetch callback with full consent detail


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_27_hwbu0i0q
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- unknown consentRequestId does not crash the handler
PASS -- fetch_consent() is never called (no hiu_id resolvable)
PASS -- ABDM still gets acked despite the missing pending record


True